# 01 — European Gas Storage & Dutch TTF Price Exploratory Analysis

## Research Context
Natural gas storage functions as the primary physical buffering mechanism for seasonal supply-demand imbalances in Europe. Wholesale benchmark pricing at the Dutch Title Transfer Facility (TTF) reflects the dynamic equilibrium between physical inventories, pipeline flows, and marginal LNG import replacement costs.

This notebook explores the raw and point-in-time aligned time-series ingested from **GIE AGSI+** (storage transparency) and **wholesale settlement records** from 2016 through 2024, verifying zero look-ahead bias and data provenance.

In [ ]:
import sys
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Connect to DuckDB analytical database
con = duckdb.connect('data/duckdb/ttf_storage.duckdb')
df_market = con.execute('SELECT * FROM aligned_daily_market ORDER BY date').fetchdf()
print(f'Loaded {len(df_market)} aligned market observations.')
df_market.head()

### 1.1 Temporal Alignment & Look-Ahead Bias Verification
Storage reports from AGSI+ are published at gas day $t$ with a 1-day reporting lag ($D-1$). To strictly avoid look-ahead bias, market trades on date $t$ must observe storage published up to date $t-1$.

In [ ]:
# Verify that market date is strictly after storage date
df_market['date'] = pd.to_datetime(df_market['date'])
df_market['storage_date'] = pd.to_datetime(df_market['storage_date'])
lag_days = (df_market['date'] - df_market['storage_date']).dt.total_seconds() / 86400.0

print(f'Minimum lag: {lag_days.min():.1f} days')
print(f'Median lag:  {lag_days.median():.1f} days')
assert (lag_days >= 1.0).all(), 'Look-ahead bias detected in alignment!'

### 1.2 Storage Fill Levels Across Gas Years
The European gas year runs from October 1 to September 30, encompassing the injection season (April–October) and withdrawal season (November–March).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_market['date'], df_market['storage_pct'], color='#1f77b4', lw=2, label='EU Storage Fill (%)')
ax.axhline(80, ls=':', color='gray', label='EU 80% Threshold')
ax.axhline(90, ls='--', color='navy', label='EU 90% Target')
ax.set_ylabel('Storage Inventory (%)')
ax.set_title('EU Aggregate Underground Gas Storage Fill Trajectory (2016–2024)')
ax.legend(loc='lower left')
plt.show()

### 1.3 TTF Wholesale Gas Price Evolution
Notice the structural shift from the tranquil pre-crisis epoch (15–25 EUR/MWh) to the 2021–2023 supply crisis, culminating in the August 2022 peak above 300 EUR/MWh.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_market['date'], df_market['ttf_price'], color='#d62728', lw=2, label='TTF Front-Month (EUR/MWh)')
ax.set_ylabel('EUR/MWh')
ax.set_title('Dutch TTF Benchmark Wholesale Natural Gas Settlement Prices')
ax.legend(loc='upper left')
plt.show()

In [ ]:
con.close()